# SensorLLM: End-to-End Training in Google Colab

This notebook sets up SensorLLM from scratch and runs a complete training pipeline using synthetic sensor data.

## What This Notebook Does

1. **Environment Setup** - Installs PyTorch, Transformers, and other dependencies
2. **Clone Repository** - Downloads the SensorLLM codebase
3. **Generate Synthetic Data** - Creates synthetic aircraft sensor data for training
4. **Train Model** - Runs a full training experiment (CNN1D encoder + Linear adapter)
5. **Results** - Shows where to find trained models and metrics

**Estimated Runtime**: 30-45 minutes on T4 GPU

**GPU Requirement**: This notebook works best with a GPU. Go to Runtime → Change runtime type → GPU


## Step 1: Setup Google Colab Environment

In [ ]:
import torch
import subprocess
from pathlib import Path

# Check GPU availability
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU available. Training will be slower. Go to Runtime → Change runtime type → GPU")

## Step 2: Clone SensorLLM Repository

**Option A: Clone from GitHub** (if public)

In [ ]:
# Clone the repository
repo_path = Path("/content/sensorllm")

if not repo_path.exists():
    print("📥 Cloning SensorLLM repository...")
    # Replace with your actual repository URL
    !git clone https://github.com/your-org/sensorllm.git /content/sensorllm
    print("✓ Repository cloned")
else:
    print(f"✓ Repository already exists at {repo_path}")

%cd /content/sensorllm

## Step 3: Install Dependencies

This installs PyTorch, Transformers, and all SensorLLM dependencies.

In [ ]:
import subprocess
import sys

print("📦 Installing SensorLLM and dependencies...")
print("This may take 5-10 minutes on first run.\n")

# Install the package in editable mode with dev dependencies
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"],
    cwd="/content/sensorllm"
)

if result.returncode == 0:
    print("✓ Installation complete")
else:
    print("✗ Installation failed")
    sys.exit(1)

## Step 4: Verify Installation

In [ ]:
# Verify all required packages
required_packages = {
    'torch': 'PyTorch',
    'transformers': 'Transformers',
    'peft': 'PEFT (LoRA)',
    'accelerate': 'Accelerate',
    'h5py': 'HDF5',
}

print("Verifying installed packages:")
for module, name in required_packages.items():
    try:
        __import__(module)
        print(f"  ✓ {name}")
    except ImportError:
        print(f"  ✗ {name} - MISSING")

print("\n✓ All packages verified")

## Step 5: Set Up Environment Variables

In [ ]:
import os
from pathlib import Path

# Create data and output directories
data_root = Path("/content/sensorllm_data")
output_root = Path("/content/sensorllm_outputs")

data_root.mkdir(parents=True, exist_ok=True)
output_root.mkdir(parents=True, exist_ok=True)

# Set environment variables
os.environ["SENSORLLM_DATA_ROOT"] = str(data_root)
os.environ["SENSORLLM_OUTPUT_ROOT"] = str(output_root)
os.environ["WANDB_MODE"] = "offline"  # Disable W&B for Colab
os.environ["HF_HUB_OFFLINE"] = "1"

print(f"Data directory: {data_root}")
print(f"Output directory: {output_root}")
print(f"W&B mode: offline (set WANDB_MODE=online to enable)")

## Step 6: Generate Synthetic Sensor Data

Creates synthetic aircraft sensor data for training. This includes:
- Vibration sensors (normal and faulty states)
- Temperature sensors
- IMU data
- Pressure sensors

You can adjust `--samples` to control dataset size (default: 20 samples per class).

In [ ]:
import subprocess
import os

print("Generating synthetic sensor data...")
print("This creates realistic time-series sensor data for training.\n")

# Generate synthetic data
# Adjust --samples for smaller/larger datasets (default: 20)
cmd = [
    "python", "scripts/generate_synthetic_data.py",
    "--data-root", str(data_root),
    "--samples", "20",  # Smaller dataset for quick testing
    "--summary"
]

result = subprocess.run(cmd, cwd="/content/sensorllm")

if result.returncode == 0:
    print("\n✓ Synthetic data generated successfully")
else:
    print("\n✗ Data generation failed")

## Step 7: Train the Model

Runs a training experiment with:
- **Encoder**: CNN1D (dilated 1D convolution)
- **Adapter**: Linear projection
- **LLM**: LLaMA 3.2 1B (small model for fast training)
- **Stage**: Alignment (freeze LLM, train encoder + adapter)

The training will:
1. Load synthetic sensor data
2. Encode signals with the CNN1D encoder
3. Project sensor embeddings to LLM token space
4. Train alignment loss
5. Save checkpoints and metrics

**Note**: First run downloads the LLaMA model (~2.5 GB). This may take 5-10 minutes.

In [ ]:
import subprocess
import os

print("🚀 Starting training...\n")
print("Training configuration:")
print("  - Encoder: CNN1D")
print("  - Adapter: Linear Projection")
print("  - LLM: LLaMA 3.2 1B")
print("  - Max steps: 500 (reduced for Colab)")
print("  - Batch size: 8")
print("\nEstimated runtime: 15-20 minutes\n")

# Build training command with Colab-optimized hyperparameters
cmd = [
    "python", "scripts/train.py",
    "--config", "configs/experiments/exp001_cnn1d_linear.yaml",
    "--override",
    "training.max_steps=500",      # Reduced for quick iteration
    "data.batch_size=8",            # Smaller batch for memory
    "training.eval_steps=50",
    "training.save_steps=100",
    "training.logging_steps=10"
]

result = subprocess.run(cmd, cwd="/content/sensorllm")

if result.returncode == 0:
    print("\n\n✓ Training completed successfully!")
else:
    print("\n\n✗ Training failed")

## Step 8: Explore Results

In [ ]:
import json
from pathlib import Path

output_root = Path("/content/sensorllm_outputs")
runs_dir = output_root / "runs"

print("Training outputs:\n")

if runs_dir.exists():
    for exp_dir in sorted(runs_dir.iterdir()):
        if exp_dir.is_dir():
            print(f"Experiment: {exp_dir.name}")
            for run_dir in sorted(exp_dir.iterdir()):
                if run_dir.is_dir():
                    print(f"  Run: {run_dir.name}")
                    
                    # List checkpoints
                    checkpoint_dirs = list(run_dir.glob("checkpoint-*"))
                    if checkpoint_dirs:
                        print(f"    Checkpoints: {len(checkpoint_dirs)}")
                    
                    # Show best model symlink
                    best_model = run_dir / "best_model"
                    if best_model.exists():
                        print(f"    Best model: {best_model.resolve()}")
                    
                    # Show config
                    config_file = run_dir / "config.yaml"
                    if config_file.exists():
                        print(f"    Config: {config_file}")
                    
                    # Show metrics
                    metrics_file = run_dir / "metrics.jsonl"
                    if metrics_file.exists():
                        print(f"    Metrics: {metrics_file}")
else:
    print("No outputs found. Check the training logs above.")

## Step 9: Plot Training Metrics

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

output_root = Path("/content/sensorllm_outputs")
runs_dir = output_root / "runs"

# Find the latest run
latest_run = None
if runs_dir.exists():
    for exp_dir in sorted(runs_dir.iterdir(), reverse=True):
        if exp_dir.is_dir():
            run_dirs = sorted(exp_dir.iterdir(), reverse=True)
            if run_dirs:
                latest_run = run_dirs[0]
                break

if latest_run is None:
    print("No training runs found yet.")
else:
    metrics_file = latest_run / "metrics.jsonl"
    
    if metrics_file.exists():
        # Load metrics
        steps = []
        train_losses = []
        eval_losses = []
        learning_rates = []
        
        with open(metrics_file) as f:
            for line in f:
                metric = json.loads(line)
                if "step" in metric:
                    steps.append(metric["step"])
                    if "loss" in metric:
                        train_losses.append(metric["loss"])
                    if "eval_loss" in metric:
                        eval_losses.append(metric["eval_loss"])
                    if "learning_rate" in metric:
                        learning_rates.append(metric["learning_rate"])
        
        # Plot results
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        
        # Training loss
        if train_losses:
            axes[0].plot(steps[:len(train_losses)], train_losses, label="Training loss", alpha=0.7)
        if eval_losses:
            # Approximate eval step indices
            eval_steps = steps[::len(steps)//len(eval_losses)] if eval_losses else []
            axes[0].plot(eval_steps[:len(eval_losses)], eval_losses, 'o-', label="Eval loss", linewidth=2)
        
        axes[0].set_xlabel("Training step")
        axes[0].set_ylabel("Loss")
        axes[0].set_title("Training and Evaluation Loss")
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
        
        # Learning rate
        if learning_rates:
            axes[1].plot(steps[:len(learning_rates)], learning_rates)
            axes[1].set_xlabel("Training step")
            axes[1].set_ylabel("Learning rate")
            axes[1].set_title("Learning Rate Schedule")
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig("/content/training_metrics.png", dpi=100, bbox_inches="tight")
        plt.show()
        
        print(f"✓ Metrics plotted and saved to /content/training_metrics.png")
    else:
        print("No metrics file found. Check training output above.")

## Step 10: Inspect Trained Model

In [ ]:
from pathlib import Path
import json

output_root = Path("/content/sensorllm_outputs")
runs_dir = output_root / "runs"

# Find the latest run
latest_run = None
if runs_dir.exists():
    for exp_dir in sorted(runs_dir.iterdir(), reverse=True):
        if exp_dir.is_dir():
            run_dirs = sorted(exp_dir.iterdir(), reverse=True)
            if run_dirs:
                latest_run = run_dirs[0]
                break

if latest_run:
    print(f"Latest run: {latest_run}\n")
    
    # Load config
    config_file = latest_run / "config.yaml"
    if config_file.exists():
        print("=" * 70)
        print("TRAINING CONFIGURATION")
        print("=" * 70)
        with open(config_file) as f:
            print(f.read())
    
    # List artifacts
    print("\n" + "=" * 70)
    print("ARTIFACTS")
    print("=" * 70)
    
    checkpoints = sorted(latest_run.glob("checkpoint-*"))
    if checkpoints:
        print(f"\nCheckpoints ({len(checkpoints)}):")
        for ckpt in checkpoints[-3:]:  # Show last 3
            print(f"  - {ckpt.name}")
    
    best_model = latest_run / "best_model"
    if best_model.exists():
        print(f"\nBest model: {best_model.resolve()}")
        model_files = list(best_model.glob("*"))
        if model_files:
            print(f"  Files: {', '.join([f.name for f in sorted(model_files)[:5]])}")
else:
    print("No training runs found.")

## Next Steps

### Run Other Experiments

Try different encoder and adapter combinations:

```python
# Transformer encoder + Q-Former adapter
!python /content/sensorllm/scripts/train.py \
  --config /content/sensorllm/configs/experiments/exp002_transformer_qformer.yaml \
  --override training.max_steps=500 data.batch_size=8

# PatchTST encoder + Perceiver adapter
!python /content/sensorllm/scripts/train.py \
  --config /content/sensorllm/configs/experiments/exp003_patchtst_perceiver.yaml \
  --override training.max_steps=500 data.batch_size=8
```

### Evaluate Your Model

```python
!python /content/sensorllm/scripts/evaluate.py \
  --config /content/sensorllm/configs/experiments/exp001_cnn1d_linear.yaml \
  --checkpoint /content/sensorllm_outputs/runs/exp001_cnn1d_linear/[DATE_TIME]/best_model
```

### Run Inference

```python
!python /content/sensorllm/scripts/infer.py \
  --checkpoint /content/sensorllm_outputs/runs/exp001_cnn1d_linear/[DATE_TIME]/best_model \
  --sensor-file /content/sensorllm_data/raw/synthetic/vibration_normal_0000.h5 \
  --prompt "Describe any anomalies in the vibration sensor data."
```

### Download Results

```python
# Download trained model and metrics
import shutil
shutil.make_archive("/content/sensorllm_results", "zip", "/content/sensorllm_outputs")
files.download("/content/sensorllm_results.zip")
```


## Troubleshooting

### Out of Memory (OOM) Error

Reduce batch size:
```python
!python scripts/train.py --config configs/experiments/exp001_cnn1d_linear.yaml \
  --override data.batch_size=4 training.max_steps=500
```

Or reduce window size:
```python
!python scripts/train.py --config configs/experiments/exp001_cnn1d_linear.yaml \
  --override data.window_size=2048 data.batch_size=8 training.max_steps=500
```

### Model Download Issues

If the model download is slow or fails, you can:
1. Use a smaller model: `meta-llama/Llama-2-7b` → `meta-llama/Llama-2-1b` (in the config)
2. Or set `HF_HOME` to cache in Google Drive for persistence

### CUDA Out of Memory

Enable gradient checkpointing in the config or reduce max_steps.

## Resources

- **Documentation**: https://github.com/your-org/sensorllm
- **Paper**: [Link to paper]
- **Configs**: See `configs/experiments/` for all available experiments
- **Data Format**: See `docs/data_format.md` for sensor data schema
